# Function calling and structured outputs with Pydantic AI

Pydantic AI makes it easy to define tools (functions) that agents can call, and to get structured outputs validated by Pydantic models.

```mermaid
sequenceDiagram
    participant User
    participant Agent
    participant LLM
    participant Tool

    User->>Agent: Question
    Agent->>LLM: Prompt + Tool definitions
    LLM->>Agent: Tool call request
    Agent->>Tool: Execute tool
    Tool->>Agent: Tool result
    Agent->>LLM: Prompt + Tool result
    LLM->>Agent: Final response
    Agent->>User: Validated output
```

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import requests
from dotenv import load_dotenv
from pydantic_ai import Agent

load_dotenv()

## Single tool

Use `@agent.tool_plain` to register a tool that doesn't need agent context.

In [ ]:
agent = Agent(
    "openai:gpt-5-nano",
    system_prompt="You're a helpful assistant. Use the tools provided when relevant.",
)


@agent.tool_plain
def get_weather(latitude: float, longitude: float) -> str:
    """Get the weather of a given latitude and longitude"""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return str(data["current"]["temperature_2m"])


response = agent.run_sync("What is the temperature in Madrid?")
print(response.output)

## Multiple tools

You can register multiple tools on an agent. The LLM will decide which ones to call.

In [ ]:
from typing import Literal

from pydantic import BaseModel


class Feedback(BaseModel):
    feedback: str
    status: Literal["OK", "REQUIRES FIXING"]


evaluator_agent = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You're a helpful assistant. Your task is to check if a given response follows the company guidelines. "
        "The company guidelines are that responses should be written in the style of a haiku. "
        "You should reply with 'OK' or 'REQUIRES FIXING' and a short explanation."
    ),
    output_type=Feedback,
)

react_agent = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You're a helpful assistant. Use the tools provided when relevant. "
        "Then draft a response and check if it follows the company guidelines. "
        "Only respond to the user after you've validated and modified the response if needed."
    ),
)


@react_agent.tool_plain
def get_weather(latitude: float, longitude: float) -> str:
    """Get the weather of a given latitude and longitude"""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return str(data["current"]["temperature_2m"])


@react_agent.tool_plain
def check_guidelines(drafted_response: str) -> Feedback:
    """Check if a given response follows the company guidelines"""
    response = evaluator_agent.run_sync(drafted_response)
    return response.output


response = react_agent.run_sync("What is the temperature in Madrid?")
print(response.output)

## Structured outputs

Pydantic AI uses Pydantic models to validate and structure the output of agents.

In [ ]:
from typing import Literal

from pydantic import BaseModel


class DocumentInfo(BaseModel):
    category: Literal["financial", "legal", "marketing", "pets", "other"]
    summary: str


agent = Agent(
    "openai:gpt-5-nano",
    system_prompt="You are a document classifier. Classify the document and provide a summary.",
    output_type=DocumentInfo,
)


def get_first_n_pages(file_path: str, n: int = 5):
    from langchain_community.document_loaders import PyPDFLoader

    loader = PyPDFLoader(file_path)
    pages = []
    for page in loader.lazy_load():
        pages.append(page)
    return "\n\n".join([p.page_content for p in pages[:n]])


document = get_first_n_pages("assets/dogs.pdf")
response = agent.run_sync(document)
print(response.output)

## Exercise

Build an agent with tools that lets users get the latest news from multiple companies and groups them according to their topic.

Use a structured output with a list of group topics and the news articles that belong to each topic.